In [36]:
import os
import json
import time
import datetime
import os.path
import asyncio
import uuid

from datetime import date, timezone
from typing import Dict, List, Optional

from dotenv import load_dotenv

# ────────────────────────────────────────────────
# Google API
# ────────────────────────────────────────────────
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

# ────────────────────────────────────────────────
# LangChain / LangGraph
# ────────────────────────────────────────────────
from langchain_core.messages import SystemMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver

# ────────────────────────────────────────────────
# UI
# ────────────────────────────────────────────────
import gradio as gr

# ────────────────────────────────────────────────
# Notebook utilities
# ────────────────────────────────────────────────
from IPython.display import Image, display

# ────────────────────────────────────────────────
# Email (SendGrid)
# ────────────────────────────────────────────────
import sendgrid
from sendgrid.helpers.mail import Mail, Email, To, Content

# ────────────────────────────────────────────────
# Local modules
# ────────────────────────────────────────────────
from agent import graph
from tools import get_recent_files_context, send_email

# ────────────────────────────────────────────────
# Environment setup
# ────────────────────────────────────────────────
load_dotenv(override=True)

# Constants
MAX_CONT = 1500

# Google OAuth credentials
CLIENT_ID = os.getenv("GOOGLE_CLIENT_ID")
CLIENT_SECRET = os.getenv("GOOGLE_SECRET_KEY")

# ────────────────────────────────────────────────
# === PUT YOUR VALUES HERE ===

In [37]:
@tool
def get_events_on_date(
    year: Optional[int] = None,
    month: Optional[int] = None,
    day: Optional[int] = None,
    max_results: int = 50
) -> list[dict]:
    """
    Fetch events on a specific date.
    If year/month/day are not provided → uses today's date.
    Use this tool when user asks about meetings, events, calendar, preparation, bootcamp etc.
    """
    if year is None or month is None or day is None:
        today = date.today()
        year = today.year
        month = today.month
        day = today.day

   
    year = year
    month = month
    day = day
    max_results  = max_results
 
    creds = None
    token_path = "token.json"
    SCOPES = ["https://www.googleapis.com/auth/calendar.readonly"]
    CLIENT_CONFIG = {
    "installed": {
        "client_id": CLIENT_ID,
        "project_id": "mycalendaragent-491108",
        "auth_uri": "https://accounts.google.com/o/oauth2/auth",
        "token_uri": "https://oauth2.googleapis.com/token",
        "auth_provider_x509_cert_url": "https://www.googleapis.com/oauth2/v1/certs",
        "client_secret": CLIENT_SECRET,
        "redirect_uris": ["http://localhost"]
    }
}

    if os.path.exists(token_path):
        creds = Credentials.from_authorized_user_file(token_path, SCOPES)

    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_config(CLIENT_CONFIG, SCOPES)
            creds = flow.run_local_server(port=0)

        with open(token_path, "w") as token_file:
            token_file.write(creds.to_json())

    try:
        service = build("calendar", "v3", credentials=creds)

        start_utc = datetime.datetime(year, month, day, 0, 0, 0, tzinfo=timezone.utc)
        end_utc = start_utc + datetime.timedelta(days=1)

        time_min = start_utc.strftime("%Y-%m-%dT%H:%M:%SZ")
        time_max = end_utc.strftime("%Y-%m-%dT%H:%M:%SZ")

        events_result = service.events().list(
            calendarId="primary",
            timeMin=time_min,
            timeMax=time_max,
            singleEvents=True,
            orderBy="startTime",
            maxResults=max_results
        ).execute()

        events = events_result.get("items", [])
        target_date = datetime.date(year, month, day)

        filtered_events = []

        for event in events:
            start_info = event.get("start", {})
            include = False

            if "dateTime" in start_info:
                dt_str = start_info["dateTime"]
                dt = datetime.datetime.fromisoformat(dt_str.replace("Z", "+00:00"))
                if dt.date() == target_date:
                    include = True

            elif "date" in start_info:
                if start_info["date"] == target_date.isoformat():
                    include = True

            if include:
                filtered_events.append({
                    "summary": event.get("summary", "(no title)"),
                    "description": event.get("description", ""),
                    "start": event.get("start", {}).get("dateTime") or event.get("start", {}).get("date"),
                    "end": event.get("end", {}).get("dateTime") or event.get("end", {}).get("date"),
                })

        return filtered_events

    except HttpError as error:
        print("Google Calendar API Error:", error)
        return []

    except Exception as e:
        print("Unexpected error:", str(e))
        return []

In [38]:
# Defining the tools
all_tools = [get_events_on_date,  get_recent_files_context, send_email]

In [29]:
from typing_extensions import TypedDict
from typing import Annotated
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START

class State(TypedDict):
    
    messages: Annotated[list, add_messages]


graph_builder = StateGraph(State)

In [ ]:

llm = ChatOpenAI(model="gpt-4o-mini")
llm_with_tools = llm.bind_tools(all_tools)


def chatbot(state: State):
    """
    Main agent logic: decides whether to call tools or respond to the user.
    Injects a strong system prompt on the first message of the conversation.
    """
    messages = state["messages"]
    
    system_prompt = """
      You are an Andela AI Engineering Bootcamp Meeting Preparation Agent.

      Your role is to help the user prepare for mentorship meetings by:
      - Summarizing recent work
      - Providing structured meeting updates
      - Generating thoughtful technical questions

      Trigger this workflow whenever the user input relates to:
      • preparation (prepare, prep, ready)
      • meetings or events
      • scheduling or planning
      • bootcamp activities

      Follow this workflow exactly:

      STEP 1 — FETCH EVENTS
      - Call: get_events_on_date
      - If no date is mentioned, call it with default settings (today).

      STEP 2 — IF NO EVENTS
      - Inform the user that no events were found for that date.
      - Ask if they want to prepare for another date.
      - Stop the workflow.

      STEP 3 — GET RECENT WORK CONTEXT
      - If events exist, call: get_recent_files_context
      - Use the default timeframe (last 24 hours).
      - Files will be Python or Markdown and represent the user's current work.

      STEP 4 — BUILD THE MEETING PREPARATION GUIDE
      Using event data and recent file context, create:

      1. Meeting Overview
      - Event title(s)
      - Time(s)
      - Likely agenda or purpose

      2. Progress Summary
      - Key recent updates
      - Current project focus

      3. Intelligent Questions
      - Provide 2 specific, technically meaningful questions for the mentor based on the work.

      STEP 5 — SEND EMAIL
      - Format the preparation guide as clean, professional HTML.
      - Call: send_email
      - Use a clear subject line.
      - Send the HTML as the email body.

      Behavior Rules:
      - Do not ask clarifying questions before calling tools when the request relates to preparation or meetings.
      - Do not describe capabilities.
      - Execute the workflow first, then provide the final response.
      - Keep communication concise and professional.
      """

    # Inject system prompt only once (at the beginning of the conversation)
    if not any(isinstance(m, SystemMessage) for m in messages):
        messages = [SystemMessage(content=system_prompt)] + messages

    # Call the model with tools enabled
    response = llm_with_tools.invoke(messages)

    return {"messages": [response]}

In [ ]:

graph_builder = StateGraph(State)
graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", ToolNode(tools=all_tools))
graph_builder.add_conditional_edges( "chatbot", tools_condition, "tools")
graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge(START, "chatbot")

memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
config = {"configurable": {"thread_id": "10"}}

async def chat(user_input: str, history):
    result = await graph.ainvoke({"messages": [{"role": "user", "content": user_input}]}, config=config)
    return result["messages"][-1].content


gr.ChatInterface(chat).launch()

In [ ]:
# Persistent thread so conversation history (and tool results) are remembered
thread_id = str(uuid.uuid4())
config = {"configurable": {"thread_id": thread_id}}

async def chat(user_input: str, history):
    """Exactly the same async flow as your example."""
    result = await graph.ainvoke(
        {"messages": [{"role": "user", "content": user_input}]},
        config=config
    )
    return result["messages"][-1].content



display(Image(graph.get_graph().draw_mermaid_png()))


In [ ]:
# Launch the Gradio chat interface
gr.ChatInterface(
    chat,
    title="Andela AI Engineering Bootcamp Meeting Prep Agent",
    description="Ask me to prepare for any meeting — I read your calendar, recent project files, and email you guides."
).launch()